# 09 — Monitor changed behavior and map the workflow to Databricks

**Plain-language question:** What can monitoring tell us before true outcomes
arrive—and what must remain the same when we move to Databricks?

**Why this matters:** production data and system behavior change. Monitoring
should trigger investigation without claiming more than the available evidence
supports.

**Estimated time:** 60–75 minutes.
**Prerequisite:** lessons 00–08; you can load the approved version and explain
its score, prediction, and release evidence.


## Preflight

Check the kernel and visibly ensure the approved local release exists.


In [ ]:
import importlib.util
import sys

required = ("mlflow", "pandas", "sklearn", "aai_local_classification")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "This notebook is using the wrong Python kernel. Close this Jupyter "
        "server, run `make notebook` from examples/local-classification, or "
        "select the 'AAI Local Classification' kernel. Missing: " + ", ".join(missing)
    )

import pandas as pd

from aai_local_classification.learning import study_root
from aai_local_classification.settings import load_settings
from aai_local_classification.tracking import local_paths

settings = load_settings()
root = study_root()
paths = local_paths(root)
print(f"✓ Python {sys.version_info.major}.{sys.version_info.minor}: {sys.executable}")
print("✓ Course imports are available")
print(f"✓ Learner state: {root}")


In [ ]:
from aai_local_classification.contracts import SplitName
from aai_local_classification.data import load_split
from aai_local_classification.inference import load_champion
from aai_local_classification.monitoring import compare_batches, shifted_batch
from aai_local_classification.workflow import (
    get_or_run_candidate_selection,
    promote_if_approved,
    run_frozen_test_gate,
)

selection = get_or_run_candidate_selection(settings, root)
decision = run_frozen_test_gate(settings, root, selection)
promotion = promote_if_approved(settings, decision, root, selection)
print(
    f"Release decision: {decision.decision.value}; registered: {promotion.get('registered')}"
)


### What you should see

`Release decision: adopt; registered: True`. If a valid reject occurred, no
champion would be loaded and inference monitoring would be skipped safely.

### Words introduced

| Word | Plain meaning | Example signal |
|---|---|---|
| input drift | Feature distribution changed | higher monthly fees |
| score drift | Model scores/actions changed | more positive predictions |
| delayed labels | Outcomes arriving after predictions | later churn truth |


## Create a transparent simulated current batch

This course has no live traffic. The helper resamples validation rows, raises
monthly fees by 10%, lowers usage by about four hours, and changes some signup
channels. It is a deliberate scenario, not evidence about real customers.

**Before you run this:** predict which mean will rise and which will fall.


In [ ]:
reference = load_split(settings, SplitName.VALIDATION, paths.data_root)
current = shifted_batch(reference, settings.random_seed + 99)

raw_comparison = pd.DataFrame(
    {
        "reference": {
            "monthly_fee_mean": reference.monthly_fee.mean(),
            "usage_hours_mean": reference.usage_hours_30d.mean(),
            "paid_search_share": (reference.signup_channel == "paid_search").mean(),
            "usage_missing_rate": reference.usage_hours_30d.isna().mean(),
        },
        "current": {
            "monthly_fee_mean": current.monthly_fee.mean(),
            "usage_hours_mean": current.usage_hours_30d.mean(),
            "paid_search_share": (current.signup_channel == "paid_search").mean(),
            "usage_missing_rate": current.usage_hours_30d.isna().mean(),
        },
    }
)
raw_comparison


### What you should see

Monthly fee rises (roughly 68.5 to the mid-70s), usage falls (roughly 28.4 to
the mid-20s), and category share changes. Exact resampling values are
deterministic for the course seed.

### How to interpret the output

Raw summaries are usually more understandable than a single drift score. They
identify what moved and in which direction; they do not tell us whether model
recall or calibration changed.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
reference.monthly_fee.plot(
    kind="hist", alpha=0.5, bins=20, ax=axes[0], label="reference"
)
current.monthly_fee.plot(kind="hist", alpha=0.5, bins=20, ax=axes[0], label="current")
reference.usage_hours_30d.plot(
    kind="hist", alpha=0.5, bins=20, ax=axes[1], label="reference"
)
current.usage_hours_30d.plot(
    kind="hist", alpha=0.5, bins=20, ax=axes[1], label="current"
)
axes[0].set_title("Monthly fee")
axes[1].set_title("Usage hours")
for axis in axes:
    axis.legend()
plt.tight_layout()


The overlapping histograms show the reference and current distributions rather
than hiding change behind one number. In production, compare appropriate
seasonal cohorts and investigate upstream product/data changes.


## Add compact diagnostics after inspecting raw values

**Population Stability Index (PSI)** summarizes binned numeric distribution
change. **Total variation (TV)** summarizes categorical-share change from 0
(same shares) to 1 (no overlap). Neither has a universal pass/fail threshold.


In [ ]:
report = compare_batches(reference, current, settings)
numeric_drift = pd.DataFrame(
    {
        "psi": pd.Series(report.numeric_psi),
        "missing_rate_change": pd.Series(report.missing_rate_delta),
    }
).sort_values("psi", ascending=False)
numeric_drift


### What you should see

Monthly fee is the largest numeric PSI value; maximum PSI is around 0.32
for this simulation. The exact number is a diagnostic to investigate alongside
the raw distributions, not an automatic declaration of failure.


In [ ]:
categorical_drift = pd.Series(
    report.categorical_total_variation,
    name="total_variation",
).sort_values(ascending=False)
categorical_drift.to_frame()


### How to interpret the output

A larger TV value means category shares moved more. It does not identify a root
cause or impact. Monitoring should link the signal to an owner and a safe
investigation playbook.


## Did model scores and actions move too?

**Before you run this:** because the simulated batch has higher fees and lower
usage, predict whether the average churn score and positive-action rate rise or
fall.


In [ ]:
def score_summary(output):
    return pd.Series(
        {
            "mean_score": output.churn_probability.mean(),
            "predicted_positive_rate": output.churn_prediction.mean(),
        }
    )


if promotion.get("registered"):
    predictor = load_champion(settings, root)
    reference_scores = predictor.predict(reference, settings)
    current_scores = predictor.predict(current, settings)
    score_comparison = pd.concat(
        [score_summary(reference_scores), score_summary(current_scores)], axis=1
    )
    score_comparison.columns = ["reference", "current"]
else:
    score_comparison = pd.DataFrame(
        {"status": ["No approved champion; score monitoring skipped."]}
    )

score_comparison


### What you should see

Mean score rises from roughly 0.17 to around 0.20, and the positive-action rate
rises from about 53% to around 59%. These values are deterministic for the
course seed.

### Misconception check

Changed inputs and scores do **not** prove degraded accuracy, recall, or
calibration. Those require correctly joined delayed churn labels. No drift also
would not prove that the system is safe or useful.


## Separate monitoring questions and owners

| Layer | Signal available now | Conclusion allowed | Not established yet |
|---|---|---|---|
| service | errors, latency, throughput | endpoint/job health changed | model quality |
| schema/data | missing/invalid fields | input contract failed | causal impact |
| inputs/scores | distributions and actions moved | investigate drift | recall/calibration loss |
| outcomes | delayed labels joined correctly | performance/calibration changed | business causality |

An alert without an owner and safe response is only telemetry.


## Map understood local objects to Databricks

The concepts stay the same; storage, identity, orchestration, and governance
become shared platform services.

| Local object you used | Databricks equivalent | New term in plain language |
|---|---|---|
| generated CSV + manifest | versioned Unity Catalog Delta table | governed table with auditable versions |
| local SQLite MLflow | hosted MLflow experiment | shared tracking service |
| local registered model | `<catalog>.<schema>.<model>` | three-part governed model name |
| local `champion` alias | Models in Unity Catalog alias | pointer on a governed model |
| Python/Make workflow | Databricks job in a Declarative Automation Bundle | reviewed declarative deployment |
| local batch predictor | job or Model Serving at a concrete version | scheduled or online inference |
| local drift report | governed tables, profiles, dashboards, alerts | shared monitoring evidence |

Read `docs/databricks-handoff.md` before adapting this project. Cloud migration
does not authorize creating infrastructure or storing credentials; use the
approved keyless identity and external platform process.


### Guided exercise

Complete one safe alert for each layer. The starter table contains a reasonable
reference answer; change the wording to match how you would explain it to an
operations partner.


In [ ]:
exercise_alerts = pd.DataFrame(
    {
        "layer": ["service", "data", "outcome"],
        "signal": [
            "elevated errors/latency",
            "required feature missing",
            "labeled recall below floor",
        ],
        "owner": ["serving owner", "data owner", "model owner"],
        "safe_action": [
            "investigate; roll back if unsafe",
            "stop batch and repair upstream",
            "review, disable, or retrain",
        ],
    }
)
exercise_alerts


**Self-check:** every row needs a measurable signal, accountable owner, and
bounded action. “Retrain automatically whenever PSI is high” is not a safe
default because drift is not proof of failure.

<details><summary>Solution explanation</summary>

Service problems belong to serving operations, broken fields to a data owner,
and labeled quality changes to model/decision owners. Each response preserves
evidence and limits harm while the cause is investigated.
</details>


In [ ]:
# Reference solution — run after your attempt
assert set(exercise_alerts.layer) == {"service", "data", "outcome"}
assert exercise_alerts.owner.str.len().gt(0).all()
assert exercise_alerts.safe_action.str.len().gt(0).all()
print("✓ Each alert has a signal, owner, and safe action")


## Recap

- Inspect raw shifts before compact diagnostics; drift prompts investigation,
  not an unsupported quality claim.
- Delayed labels are required to measure post-release recall and calibration.
- Databricks changes the platform implementation, not the evidence chain you
  practiced locally.

**Evidence used:** registered model version, alias, threshold, reference batch,
simulated current batch, input/score diagnostics, and an operating plan. The
shifted batch is created in memory and is not production data.

**Course completion check:** you can trace one prediction back to a concrete
model version, threshold, input contract, selected candidate, validation
evidence, frozen-test decision, dataset fingerprint, source/dependency evidence,
and monitoring owner.

Continue with `docs/glossary.md`, `docs/resources.md`, and
`docs/databricks-handoff.md`. Run `make check` only when you want the full
contributor verification gate.
